# 3A · Vietnamese forest bird calls — solution notebook
Multi-label (10 species) on 4 s / 8 kHz clips; metric = macro ROC-AUC. Train = *focal* recordings, test = *soundscapes*.
The whole method is: **make the training data look like the test data**, validate on soundscape-like mixtures, then a small SED-CNN.

Code lives in the modules next to this notebook (`common.py`, `mix.py`, `train.py`, `predict.py`, `prior.py`); this notebook walks through them.
Set `CACHE_3A` to choose where the ~200 MB feature cache goes. A GPU is used automatically if present.

## 0. Setup

In [ ]:
import os, sys
os.environ.setdefault('THREADS_3A', '2')
sys.path.insert(0, '.')
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from common import *          # paths, SPECIES, mel front-end, BirdSED model, macro_auc
from mix import Mixer, distractor
print(DEVICE, SPECIES)

## 1. Data (the part that matters most)
**What we saw when looking at spectrograms**
* Train clips are loud, clean focal recordings (one bird, high SNR). Test soundscapes are ~4x quieter relative to their noise and contain
  stationary insect/cicada bands (2–3.8 kHz), pulsed insect tones (3–3.9 kHz), engine hum harmonics, rain, large harmonic arches (non-target birds),
  and a very common **frog chorus** (regular pulse train with 1–3 harmonic dots) that a naive model mistakes for *silphe*.
* Test clips hold 0–3 species, the train clips mostly one (+13 % with secondary labels).
* Metadata: train hours suggest strong diurnal priors (owl at night, most species 5–17 h) — but applying them to the test **hurt** (see README), so `hour` is not used.

**Recipe**
1. Cache log-mel *power* (64 mels, 50–4000 Hz, hop 16 ms) with every clip RMS-normalised.
2. *Denoise* focal clips by subtracting each mel bin's median over time → clean bird "templates".
3. Backgrounds = the stationary floor (20th percentile per bin) of a random **unlabelled test clip**, times random texture and slow gain drift.
4. Add Poisson(1.5) procedural *distractors* (insect tones, arches, hum, rain, cicada band, clicks, frog chorus).
5. Add 0–3 bird templates (class-balanced sampling), random circular time shift, ±2 mel-bin pitch shift, SNR −10…+12 dB; labels = union (secondary labels included).
6. Features: 2 channels = clip-median-centred log-mel and per-bin-median-removed log-mel (a cheap PCEN-like normalisation that removes stationary noise).

In [ ]:
d = load_cache()
tr, pu, pr = read_csvs()
y = train_targets(tr)
birds = denoise(d['train'])
floors = stationary_floor(np.concatenate([d['public'], d['private']]))
mixer = Mixer(birds, y, floors, seed=0)
X, Y = mixer.batch(8)
fig, ax = plt.subplots(2, 4, figsize=(16, 5))
for a, x, t in zip(ax.flat, X, Y):
    a.imshow(np.log(x.numpy() + 1e-6), origin='lower', aspect='auto'); a.set_title(','.join(np.array(SPECIES)[t.numpy() > 0]) or 'empty')
plt.tight_layout()

**Validation that mimics the test.** 20 % of focal clips (stratified) are held out; 800 fixed mixtures of those clips over
backgrounds from *other* test clips than the ones used for training backgrounds. A random split of focal clips scores ~1.0 and is useless.

## 2. Model
`BirdSED`: light VGG-style CNN (BatchNorm, 7 conv layers, width 32→256, stride-2 stem) on the 2-channel log-mel, frequency pooled (mean+max),
then a **SED attention head**: frame-wise logits, softmax-attention over time, averaged with the max frame logit. ~1.2 M params.

In [ ]:
m = BirdSED(w=32); print(sum(p.numel() for p in m.parameters()), m(torch.zeros(2, 2, 64, 251)).shape)

## 3. Training & inference
* AdamW (wd 1e-2), OneCycle LR 2e-3, 40 "epochs" × 1600 freshly synthesised mixtures, batch 32, BCE, SpecAugment (2 freq + 2 time masks).
* Checkpoint = best soundscape-validation AUC in the second half (full-data models: last-epoch-ish best on the same val).
* **Round 2 (pseudo-labels):** the round-1 model labels the 800 test clips (soft, p<0.1→0); round-2 models use real test clips as backgrounds 50 % of the time,
  with the pseudo-labels as targets — so the model sees the *real* frogs/insects/rain.
* Inference: 4 circular time-shift TTA, average of 5 models (hold-out model, 2 full-data seeds, 2 pseudo-label seeds).

Training one model (`python train.py --name X --epochs 40 --per_epoch 1600 --width 32 [--full] [--pseudo FILE]`) takes ~6 min on a GTX 1050 Ti
and ~30+ min on 2 CPU threads, so the cell below is a short demo; `solution.py` runs the full pipeline.

In [ ]:
!python train.py --name demo --epochs 2 --per_epoch 320 --width 16

## 4. Predict (full pipeline → `public_submission.csv`, `private_submission.csv`)

In [ ]:
# Full end-to-end run (trains 5 models, pseudo-labels, ensembles, writes the CSVs here):
# !python solution.py
# If the checkpoints already exist in CACHE_3A, only the ensemble step is needed:
# !python predict.py R1_full F1 F2 PF1 PF2 --tta 4
sub = pd.read_csv('public_submission.csv'); assert list(sub.columns) == ['id', *SPECIES]; sub.head()